# 다목적 최적화와 파레토 전선 실습

**Multi-objective Optimization · Pareto Front · 파레토 최적**

상충하는 여러 목표를 동시에 고려해 더 나빠지지 않고는 개선할 수 없는 해 집합을 찾는 최적화.

소재 분야에서 이해하기: 강도와 연성을 함께 고려한 후보 집합을 뽑는다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [pymoo 다목적 최적화 문서](https://pymoo.org/)

## 1. 상충하는 두 목표

강도와 연성을 함께 높이려 할 때 어떤 후보가 남는지 봅니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

n = 400
composition = rng.uniform(0, 1, n)
process = rng.uniform(0, 1, n)
strength = 200 + 300 * composition - 60 * process + rng.normal(0, 8, n)
ductility = 40 - 25 * composition + 18 * process + rng.normal(0, 1.5, n)
plt.scatter(strength, ductility, s=12)
plt.xlabel('strength (MPa)'); plt.ylabel('elongation (%)'); plt.show()

In [ ]:
def pareto_front(objectives):
    """모든 목표를 최대화할 때 지배되지 않는 점의 인덱스."""
    keep = np.ones(len(objectives), bool)
    for index, point in enumerate(objectives):
        if not keep[index]:
            continue
        dominated = np.all(objectives <= point, axis=1) & np.any(objectives < point, axis=1)
        keep[dominated] = False
    return np.flatnonzero(keep)

objectives = np.column_stack([strength, ductility])
front = pareto_front(objectives)
order = front[np.argsort(strength[front])]
plt.scatter(strength, ductility, s=10, c='lightgray', label='all candidates')
plt.plot(strength[order], ductility[order], 'ro-', ms=5, label='Pareto front')
plt.xlabel('strength (MPa)'); plt.ylabel('elongation (%)'); plt.legend(); plt.show()
print('후보 %d개 중 파레토 전선 %d개' % (n, len(front)))

## 2. 가중합은 전선의 일부만 봅니다

In [ ]:
for weight in (0.1, 0.5, 0.9):
    scalarised = weight * (strength / strength.max()) + (1 - weight) * (ductility / ductility.max())
    pick = int(np.argmax(scalarised))
    print('가중치 %.1f -> 강도 %.0f MPa, 연신율 %.1f%%' % (weight, strength[pick], ductility[pick]))
print('\n가중합은 가중치 하나마다 해 하나만 줍니다. 전선을 보고 나서 목표를 정하는 편이 정보가 많습니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#multi-objective)을 여세요.